# Evaluating the CareConnect SDoH Agent
### *Agentic AI for Digital Health* — the evals lesson

Memory, "dreaming," autonomous tool use — every technique for making an agent better is a form of **hill-climbing**. None of it is safe without an eval to tell you whether a change went *up* the hill or *off a cliff*. **The eval is the hill.**

This notebook evaluates the `careconnect-sdoh` agent across **five layers**, surfaces **three real defects** the polished demo hides, then applies the fixes and watches the metrics move.

> Runs offline against a deterministic `mock` agent — no API key needed. Set `CARECONNECT_EVAL_PROVIDER=anthropic|openai` to run against a real model. See `EVALS.md` for the full write-up.

## 0. Setup
The harness is pure-Python and provider-agnostic. The deterministic tools are ported faithfully from `CareConnect.Tools.SDoHToolSet` (ObjectScript) so we can evaluate them with zero infrastructure.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # the evals/ directory

from careconnect_evals import run_suite, print_report, scorers
from careconnect_evals.tools_local import LocalToolClient, PATIENTS
from careconnect_evals.improved import assess_improved, draft_care_plan_improved

EVALS_DIR = Path.cwd().parent
cases = json.load(open(EVALS_DIR / "golden_cases.json"))["cases"]
print(f"Loaded {len(cases)} golden cases:", [c["id"] for c in cases])

## 1. The system under test is a *hybrid* — and evals live at the seams

```
  LLM orchestration  ->  deterministic tools  ->  real side effects
  (which tool, what      (rule-based scoring,     (IRIS Interop workflow,
   order, what args)      care planning)           audit trail in MessageHeader)
```

The risk-scoring tool *looks* deterministic, but it keyword-matches on a clinical summary **the LLM writes**. That seam is exactly where variance hides — and where most demos never look.

## 2. Two ground truths per case

Every golden case carries **`rule_expected`** (what the shipped tool *should* output — a regression target) **and** **`human_label`** (what's *actually* true for the patient, per a clinician). The gap between them is the whole point: a green regression test does **not** mean the spec is right.

In [ ]:
james = next(c for c in cases if c["id"] == "james-complete")
print(json.dumps({k: james[k] for k in ["id", "rule_expected", "human_label", "tags"]}, indent=2))

Note James: the **rule** says Economic = LOW and priority = HIGH. A **clinician** reading *"skipping medications due to cost, subsidized housing"* says Economic = HIGH and priority = URGENT. The rule will pass its own regression test while under-calling a real, urgent need.

## 3. Run the whole suite (offline mock agent)

In [ ]:
report = run_suite("mock")
print_report(report)

## 4. Layer by layer

### L1 — The "deterministic" scorer isn't, end-to-end
Same patient (Maria), two phrasings of the *same facts*. The shipped tool keyword-matches, so a faithful paraphrase silently collapses her priority.

In [ ]:
client = LocalToolClient()
canonical = client.FetchPatientSummary("maria-gonzalez-001")
paraphrase = next(c for c in cases if c["adversarial"])["agent_summary"]

print("CANONICAL note  ->", client.AssessSDoHRisk("maria-gonzalez-001", canonical).splitlines()[-1])
print("PARAPHRASED note->", client.AssessSDoHRisk("maria-gonzalez-001", paraphrase).splitlines()[-1])
print("\nLesson: pin the deterministic core with exact-match (L1), but remember its INPUT is LLM-generated.")

### L1b — The rule has clinical blind spots (recall vs a clinician)
Exact-match against the spec is green; precision/recall against *clinician truth* tells the real story.

In [ ]:
print(f"{'patient':<22}{'recall':>7}  {'missed (real needs)':<28}{'rule pri / clinician pri'}")
for r in report["results"]:
    if r["adversarial"]:
        continue
    L = r["layers"]["L1b_vs_human"]
    print(f"{r['patientId']:<22}{L['recall']:>7}  {str(L['missed_domains']):<28}{L['priority_got']} / {L['priority_human']}")
s = report["summary"]
print(f"\nMicro recall = {s['human_truth_micro_recall']}  precision = {s['human_truth_micro_precision']}")
print("Precision 1.0 but recall 0.75 -> the rule never cries wolf, but misses 1-in-4 genuine needs.")

### L2 — Trajectory: evaluate the *path*, not just the answer
Did the agent call the right tools, in an order that respects the workflow's constraints (assess before plan; start production before triggering)?

In [ ]:
r = report["results"][0]
print("tools:", r["tool_trajectory"])
print(json.dumps(r["layers"]["L2_trajectory"], indent=2))

### L3 — Outcome: the audit trail *is* the ground truth
Did the follow-up workflow actually fire when (and only when) it should? We read the (simulated) `Ens.MessageHeader` trail — no scraping of model text.

In [ ]:
for r in report["results"]:
    L = r["layers"]["L3_outcome"]
    print(f"{r['id']:<32} fired={str(L['follow_up_fired']):<6} expected={str(L['expected']):<6} audit_msgs={L['audit_trail_messages']}  {'PASS' if L['passed'] else 'FAIL'}")

### L5 — Cross-case: are the care plans actually personalized?
This defect is invisible in any single case. `DraftCarePlan` branches on the *presence of domain names* in the scores string — and those names are always present. So...

In [ ]:
plans = {r["patientId"]: r["care_plan"] for r in report["results"] if not r["adversarial"]}
bodies = {p: pl.split("\n", 1)[1].strip() for p, pl in plans.items()}
print(f"{len(set(bodies.values()))} distinct plan body across {len(bodies)} patients with different risk profiles.\n")
print(list(plans.values())[0])
print("\n^ every patient receives this exact plan. The 'personalization' is an illusion.")

## 5. Visualize the report

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

layers = list(report["summary"]["per_layer"].keys())
rates = [v["passed"] / v["total"] for v in report["summary"]["per_layer"].values()]
colors = ["#d9534f" if x < 1 else "#5cb85c" for x in rates]
ax1.barh(layers, rates, color=colors)
ax1.set_xlim(0, 1); ax1.set_xlabel("pass rate"); ax1.set_title("Layer pass rates (shipped tools)")
ax1.invert_yaxis()

pats = [r["patientId"].split("-")[0].title() for r in report["results"] if not r["adversarial"]]
recs = [r["layers"]["L1b_vs_human"]["recall"] for r in report["results"] if not r["adversarial"]]
ax2.bar(pats, recs, color="#f0ad4e")
ax2.axhline(1.0, ls="--", c="gray"); ax2.set_ylim(0, 1.1)
ax2.set_ylabel("clinician recall"); ax2.set_title("Genuine needs the rule flags (per patient)")
plt.tight_layout(); plt.show()

## 6. Close the loop — the *improve* half

`careconnect_evals/improved.py` holds the fixes the evals justify:
- **Fix #1:** broaden the risk lexicon (`cost`, `uninsured`, `food insecure`, `skipping meds`, ...)
- **Fix #2:** make `DraftCarePlan` read domain **values** (HIGH), not names

Re-run the same metrics against the fixed tools:

In [ ]:
tp = fn = 0
impr_plans, impr_profiles = {}, {}
for c in cases:
    if c["adversarial"]:
        continue
    pid = c["patientId"]
    summary = client.FetchPatientSummary(pid)
    scored = assess_improved(pid, summary)
    counts = scorers.score_against_human(scored, c["human_label"])["_counts"]
    tp += counts["tp"]; fn += counts["fn"]
    impr_plans[pid] = draft_care_plan_improved(pid, scored)
    impr_profiles[pid] = tuple(sorted(
        k for k, v in scorers.parse_assessment(scored)["domains"].items()
        if v == "HIGH" and k in scorers.PLAN_DOMAINS))

print(f"clinician recall: 0.75  ->  {round(tp / (tp + fn), 3)}")
diff = scorers.score_care_plan_differentiation(impr_plans, impr_profiles)
print(f"care-plan differentiation: FAIL  ->  {'PASS' if diff['passed'] else 'FAIL'}  ({diff['distinct_plans']} distinct plans)")
james_summary = client.FetchPatientSummary("james-okafor-002")
print(f"\nJames priority: HIGH  ->  {scorers.parse_assessment(assess_improved('james-okafor-002', james_summary))['priority']}")

**That is the entire discipline in one picture:** define the metric -> run it -> read the failures -> make the smallest change that moves the number -> re-run. `run_evals.py` exits non-zero on any L1/L2/L3 regression, so this is the CI gate on every prompt or tool change.

## 7. The "dreaming" connection & where this goes next

The ChatGPT-memory post's *dreaming* is offline self-improvement — the model reflecting on synthetic experience between sessions. The prerequisite nobody headlines is an **eval harness**: you cannot let a system rewrite its own behavior unless you can measure whether each rewrite helped.

The golden set here is the seed. The natural next step — sketched, not built — is **synthetic case generation**: prompt an LLM to invent new patients with known ground-truth SDoH flags (especially adversarial paraphrases), grow coverage beyond three hand-written patients, and catch regressions a human would never think to write. *The model helps build its own exam; the harness keeps it honest.*

**Takeaways for the slide:**
1. You can't improve what you can't measure — evals are the unit tests of behavior.
2. Pin the deterministic core; the variance hides in the seam with the LLM.
3. In agents, evaluate the *path*, not just the answer.
4. Build on auditable systems — the IRIS Interop trace is a free outcome oracle.
5. "Matches spec" != "is correct" — keep a human ground truth.
6. Close the loop: measure -> fix -> re-measure, and gate it in CI.